In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from imblearn.over_sampling import SMOTE
from sklearn.feature_selection import SelectKBest, f_classif
import warnings
warnings.filterwarnings('ignore')
import os
import sys
sys.path.append(os.path.abspath(".."))
sys.path.append(os.path.abspath("../src"))
import pickle

***1. Load the cleaned dataset***

In [2]:
df = pd.read_csv("D:\HOPE AI\Capstone Project\Telecom Churn Prediction\Datasets\Preprocessed\Cleaned_telco_churn.csv")
df.head()

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,OnlineSecurity,OnlineBackup,DeviceProtection,...,MonthlyCharges,TotalCharges,Churn,InternetService_Fiber optic,InternetService_No,Contract_One year,Contract_Two year,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check
0,1,0,1,0,1,0,0,0,1,0,...,29.85,29.85,0,0,0,0,0,0,1,0
1,0,0,0,0,34,1,0,1,0,1,...,56.95,1889.50,0,0,0,1,0,0,0,1
2,0,0,0,0,2,1,0,1,1,0,...,53.85,108.15,1,0,0,0,0,0,0,1
3,0,0,0,0,45,0,0,1,0,1,...,42.30,1840.75,0,0,0,1,0,0,0,0
4,1,0,0,0,2,1,0,0,0,0,...,70.70,151.65,1,1,0,0,0,0,1,0


***2. Seperating feature and target variables***

In [3]:
independent = df.drop(columns=["Churn"])
dependent = df["Churn"]

In [4]:
independent.head()

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,OnlineSecurity,OnlineBackup,DeviceProtection,...,PaperlessBilling,MonthlyCharges,TotalCharges,InternetService_Fiber optic,InternetService_No,Contract_One year,Contract_Two year,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check
0,1,0,1,0,1,0,0,0,1,0,...,1,29.85,29.85,0,0,0,0,0,1,0
1,0,0,0,0,34,1,0,1,0,1,...,0,56.95,1889.50,0,0,1,0,0,0,1
2,0,0,0,0,2,1,0,1,1,0,...,1,53.85,108.15,0,0,0,0,0,0,1
3,0,0,0,0,45,0,0,1,0,1,...,0,42.30,1840.75,0,0,1,0,0,0,0
4,1,0,0,0,2,1,0,0,0,0,...,1,70.70,151.65,1,0,0,0,0,1,0


In [5]:
dependent.head()

0    0
1    0
2    1
3    0
4    1
Name: Churn, dtype: int64

***3. Splitting training and testing data***

In [6]:
X_train, X_test, Y_train, Y_test = train_test_split(independent, dependent, test_size=0.2, random_state=0)

In [7]:
X_train.shape

(5634, 23)

In [8]:
X_test.shape

(1409, 23)

In [9]:
Y_train.shape

(5634,)

In [10]:
Y_test.shape

(1409,)

***4. Standard scaling on continuous features***

In [11]:
num_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']
sc = StandardScaler()

X_train[num_cols] = sc.fit_transform(X_train[num_cols])
X_test[num_cols] = sc.transform(X_test[num_cols])

***5. Apply SelectK best Feature Selection***

In [12]:
from Feature_selection import Feature_Selectors
X_train_k, X_test_k = Feature_Selectors.selectkbest(X_train, X_test, Y_train, 7)


In [13]:
print("Selected Training Features Shape:", X_train_k.shape)
print("Selected Testing Features Shape:", X_test_k.shape)

Selected Training Features Shape: (5634, 7)
Selected Testing Features Shape: (1409, 7)


***6. Churn value Balancing***

In [14]:
# Checking the balance
Y_train.value_counts()

Churn
0    4133
1    1501
Name: count, dtype: int64

In [15]:
# SMOTE for balancing the data
smote = SMOTE(random_state=42)
X_train_k_resampled, Y_train_resampled = smote.fit_resample(X_train_k, Y_train)

In [16]:
# Checking after smote
Y_train_resampled.value_counts()

Churn
0    4133
1    4133
Name: count, dtype: int64

***7. Model Training***

In [17]:
from Models import Classification_Models
from Evaluation_Metrics import ModelEvaluator

In [18]:
# Logistic Regression
print("Logistic Regression Model")
y_pred_lr , grid_lr = Classification_Models.logistic(X_train_k_resampled,Y_train_resampled,X_test_k)
best_lr, cm_lr, report_lr, roc_lr = ModelEvaluator.evaluate(y_pred_lr, grid_lr, X_test_k, Y_test)

print("Best Parameters:", best_lr)
print("Confusion Matrix:\n", cm_lr)
print("Classification Report:\n", report_lr)
print(f"ROC-AUC Score: {roc_lr:.4f}")

Logistic Regression Model
Fitting 5 folds for each of 16 candidates, totalling 80 fits
Best Parameters: {'C': 1, 'penalty': 'l2', 'solver': 'saga'}
Confusion Matrix:
 [[743 298]
 [ 88 280]]
Classification Report:
               precision    recall  f1-score   support

           0       0.89      0.71      0.79      1041
           1       0.48      0.76      0.59       368

    accuracy                           0.73      1409
   macro avg       0.69      0.74      0.69      1409
weighted avg       0.79      0.73      0.74      1409

ROC-AUC Score: 0.8205


In [19]:
# Support vector machine
print("SVM Model")
y_pred_svm, grid_svm = Classification_Models.SVM(X_train_k_resampled, Y_train_resampled, X_test_k)
best_svm, cm_svm, report_svm, roc_svm = ModelEvaluator.evaluate(y_pred_svm, grid_svm, X_test_k, Y_test)

print("Best Parameters:", best_svm)
print("Confusion Matrix:\n", cm_svm)
print("Classification Report:\n", report_svm)
print(f"ROC-AUC Score: {roc_svm:.4f}")

SVM Model
Fitting 5 folds for each of 12 candidates, totalling 60 fits
Best Parameters: {'C': 10, 'gamma': 'scale', 'kernel': 'rbf'}
Confusion Matrix:
 [[726 315]
 [ 81 287]]
Classification Report:
               precision    recall  f1-score   support

           0       0.90      0.70      0.79      1041
           1       0.48      0.78      0.59       368

    accuracy                           0.72      1409
   macro avg       0.69      0.74      0.69      1409
weighted avg       0.79      0.72      0.74      1409

ROC-AUC Score: 0.7952


In [20]:
# K-Nearest neighbour
print("KNN  Model")
y_pred_knn, grid_knn = Classification_Models.KNN(X_train_k_resampled, Y_train_resampled, X_test_k)
best_knn, cm_knn, report_knn, roc_knn = ModelEvaluator.evaluate(y_pred_knn, grid_knn, X_test_k, Y_test)

print("Best Parameters:", best_knn)
print("Confusion Matrix:\n", cm_knn)
print("Classification Report:\n", report_knn)
print(f"ROC-AUC Score: {roc_knn:.4f}")

KNN  Model
Fitting 5 folds for each of 24 candidates, totalling 120 fits
Best Parameters: {'algorithm': 'auto', 'n_neighbors': 7, 'weights': 'distance'}
Confusion Matrix:
 [[776 265]
 [134 234]]
Classification Report:
               precision    recall  f1-score   support

           0       0.85      0.75      0.80      1041
           1       0.47      0.64      0.54       368

    accuracy                           0.72      1409
   macro avg       0.66      0.69      0.67      1409
weighted avg       0.75      0.72      0.73      1409

ROC-AUC Score: 0.7663


In [21]:
# Naive Bayes 
print("Naive Bayes Model")
y_pred_nb, grid_nb = Classification_Models.NaiveBayes(X_train_k_resampled, Y_train_resampled, X_test_k)
best_nb, cm_nb, report_nb, roc_nb = ModelEvaluator.evaluate(y_pred_nb, grid_nb, X_test_k, Y_test)

print("Best Parameters:", best_nb)
print("Confusion Matrix:\n", cm_nb)
print("Classification Report:\n", report_nb)
print(f"ROC-AUC Score: {roc_nb:.4f}")

Naive Bayes Model
Fitting 5 folds for each of 20 candidates, totalling 100 fits
Best Parameters: {'var_smoothing': np.float64(0.11288378916846892)}
Confusion Matrix:
 [[639 402]
 [ 69 299]]
Classification Report:
               precision    recall  f1-score   support

           0       0.90      0.61      0.73      1041
           1       0.43      0.81      0.56       368

    accuracy                           0.67      1409
   macro avg       0.66      0.71      0.65      1409
weighted avg       0.78      0.67      0.69      1409

ROC-AUC Score: 0.8051


In [22]:
#Decision Tree
print("Decision Tree Model")
y_pred_dt, grid_dt = Classification_Models.DecisionTree(X_train_k_resampled, Y_train_resampled, X_test_k)
best_dt, cm_dt, report_dt, roc_dt = ModelEvaluator.evaluate(y_pred_dt, grid_dt, X_test_k, Y_test)

print("Best Parameters:", best_dt)
print("Confusion Matrix:\n", cm_dt)
print("Classification Report:\n", report_dt)
print(f"ROC-AUC Score: {roc_dt:.4f}")

Decision Tree Model
Fitting 5 folds for each of 8 candidates, totalling 40 fits
Best Parameters: {'criterion': 'gini', 'max_features': 'sqrt', 'splitter': 'best'}
Confusion Matrix:
 [[813 228]
 [176 192]]
Classification Report:
               precision    recall  f1-score   support

           0       0.82      0.78      0.80      1041
           1       0.46      0.52      0.49       368

    accuracy                           0.71      1409
   macro avg       0.64      0.65      0.64      1409
weighted avg       0.73      0.71      0.72      1409

ROC-AUC Score: 0.6534


In [23]:
# Random Forest
print("Random Forest Model")
y_pred_rf, grid_rf = Classification_Models.RandomForest(X_train_k_resampled, Y_train_resampled, X_test_k)
best_rf, cm_rf, report_rf, roc_rf = ModelEvaluator.evaluate(y_pred_rf, grid_rf, X_test_k, Y_test)

print("Best Parameters:", best_rf)
print("Confusion Matrix:\n", cm_rf)
print("Classification Report:\n", report_rf)
print(f"ROC-AUC Score: {roc_rf:.4f}")

Random Forest Model
Fitting 5 folds for each of 12 candidates, totalling 60 fits
Best Parameters: {'class_weight': 'balanced_subsample', 'criterion': 'entropy', 'n_estimators': 50}
Confusion Matrix:
 [[823 218]
 [163 205]]
Classification Report:
               precision    recall  f1-score   support

           0       0.83      0.79      0.81      1041
           1       0.48      0.56      0.52       368

    accuracy                           0.73      1409
   macro avg       0.66      0.67      0.67      1409
weighted avg       0.74      0.73      0.74      1409

ROC-AUC Score: 0.7765


In [24]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import pandas as pd

eval_records = [
    {"Model": "Logistic Regression", "Accuracy": accuracy_score(Y_test, y_pred_lr), "Precision": precision_score(Y_test, y_pred_lr, zero_division=0), "Recall": recall_score(Y_test, y_pred_lr, zero_division=0), "F1-Score": f1_score(Y_test, y_pred_lr, zero_division=0), "ROC-AUC": roc_lr, "Best Params": str(best_lr)},
    {"Model": "SVM", "Accuracy": accuracy_score(Y_test, y_pred_svm), "Precision": precision_score(Y_test, y_pred_svm, zero_division=0), "Recall": recall_score(Y_test, y_pred_svm, zero_division=0), "F1-Score": f1_score(Y_test, y_pred_svm, zero_division=0), "ROC-AUC": roc_svm, "Best Params": str(best_svm)},
    {"Model": "KNN", "Accuracy": accuracy_score(Y_test, y_pred_knn), "Precision": precision_score(Y_test, y_pred_knn, zero_division=0), "Recall": recall_score(Y_test, y_pred_knn, zero_division=0), "F1-Score": f1_score(Y_test, y_pred_knn, zero_division=0), "ROC-AUC": roc_knn, "Best Params": str(best_knn)},
    {"Model": "Naive Bayes", "Accuracy": accuracy_score(Y_test, y_pred_nb), "Precision": precision_score(Y_test, y_pred_nb, zero_division=0), "Recall": recall_score(Y_test, y_pred_nb, zero_division=0), "F1-Score": f1_score(Y_test, y_pred_nb, zero_division=0), "ROC-AUC": roc_nb, "Best Params": str(best_nb)},
    {"Model": "Decision Tree", "Accuracy": accuracy_score(Y_test, y_pred_dt), "Precision": precision_score(Y_test, y_pred_dt, zero_division=0), "Recall": recall_score(Y_test, y_pred_dt, zero_division=0), "F1-Score": f1_score(Y_test, y_pred_dt, zero_division=0), "ROC-AUC": roc_dt, "Best Params": str(best_dt)},
    {"Model": "Random Forest", "Accuracy": accuracy_score(Y_test, y_pred_rf), "Precision": precision_score(Y_test, y_pred_rf, zero_division=0), "Recall": recall_score(Y_test, y_pred_rf, zero_division=0), "F1-Score": f1_score(Y_test, y_pred_rf, zero_division=0), "ROC-AUC": roc_rf, "Best Params": str(best_rf)}
]

df_selectk_results = pd.DataFrame(eval_records).sort_values(by=["ROC-AUC", "Recall"], ascending=False).reset_index(drop=True)
display(df_selectk_results.round(4))

,Model,Accuracy,Precision,Recall,F1-Score,ROC-AUC,Best Params
0,Logistic Regression,0.7260,0.4844,0.7609,0.5920,0.8205,"{'C': 1, 'penalty': 'l2', 'solver': 'saga'}"
1,Naive Bayes,0.6657,0.4265,0.8125,0.5594,0.8051,{'var_smoothing': np.float64(0.112883789168468...
2,SVM,0.7189,0.4767,0.7799,0.5918,0.7952,"{'C': 10, 'gamma': 'scale', 'kernel': 'rbf'}"
3,Random Forest,0.7296,0.4846,0.5571,0.5183,0.7765,"{'class_weight': 'balanced_subsample', 'criter..."
4,KNN,0.7168,0.4689,0.6359,0.5398,0.7663,"{'algorithm': 'auto', 'n_neighbors': 7, 'weigh..."
5,Decision Tree,0.7133,0.4571,0.5217,0.4873,0.6534,"{'criterion': 'gini', 'max_features': 'sqrt', ..."


***Amoug all the models Logistic regression model has performed well across all the tested metrics***

***It has the highest ROC-AUC score and F1 score and nearly best accuracy***

***Logistic regression is the best model for this problem statement***

In [25]:
# Geetting the selected 7 features names
selector_7 = SelectKBest(score_func=f_classif, k=7)
selector_7.fit(X_train, Y_train)

# Get the list of 7 selected column names
selected_7_features = list(X_train.columns[selector_7.get_support()])
print("Selected 7 Features:", selected_7_features)

Selected 7 Features: ['tenure', 'PaperlessBilling', 'TotalCharges', 'InternetService_Fiber optic', 'InternetService_No', 'Contract_Two year', 'PaymentMethod_Electronic check']


***8. Testing the model on new unseen data***

In [26]:
tenure = int(input("enter tenure in months"))
PaperlessBilling = int(input("Customer has paperlessbilling:yes=1,no=0"))
TotalCharges = float(input("Enter customer Total charges"))
InternetService_Fiber_optic = int(input("Customer has fiber optic service:yes=1,no=0"))
InternetService_No = int(input("Customer has internet service:yes=1,no=0"))
Contract_Two_year = int(input("Customer contract is 2 year:yes=1,no=0"))
PaymentMethod_Electronic_check = int(input("Customer payment is electronic checkr:yes=1,no=0"))

In [27]:
user_input = {
    'tenure': tenure,
    'PaperlessBilling': PaperlessBilling,
    'TotalCharges': TotalCharges,
    'InternetService_Fiber optic': InternetService_Fiber_optic,
    'InternetService_No': InternetService_No,
    'Contract_Two year': Contract_Two_year,
    'PaymentMethod_Electronic check': PaymentMethod_Electronic_check
}

In [28]:
user_df = pd.DataFrame([user_input])
scaler = StandardScaler()
num_col = ['tenure','TotalCharges']

user_df[num_col] = scaler.fit_transform(user_df[num_col])

In [29]:
future_prediction = grid_lr.predict(user_df)
print(f"Customer churn prediction = {future_prediction[0]}")

Customer churn prediction = 0


***9. Save the best model***

In [31]:
# Path to save the best model
save_dir = r"D:\HOPE AI\Capstone Project\Telecom Churn Prediction\Saved Models"
os.makedirs(save_dir, exist_ok=True)

# 1. Save the best model (Logistic Regression)
pickle.dump(grid_lr.best_estimator_, open(os.path.join(save_dir, "Customer_Churn_Prediction_Best_Model_7.pkl"), "wb"))

# 2. Save the standard scaler
pickle.dump(scaler, open(os.path.join(save_dir, "scaler_7.pkl"), "wb"))

# 3. Save the fitted SelectKBest selector (k=7)
pickle.dump(selector_7, open(os.path.join(save_dir, "selectk_7.pkl"), "wb"))

# 4. Save the 7 selected feature column names 
pickle.dump(selected_7_features, open(os.path.join(save_dir, "selected_7_features.pkl"), "wb"))
